# ![Machine Learning Lab](banner.jpg)

# Laboratorio 11 - Actividad

## Instrucciones generales

1. Esta actividad es de carácter individual. No se permite entregar la actividad después de la fecha establecida.
2. Al responder las preguntas de las actividades, por favor marquen las respuestas con la sección a la que corresponden, por ejemplo: `## 1.1 Descarga de datos`. Es preferible que esto lo hagan con secciones de MarkDown.
3. Por favor asegurarse de que el notebook entregado tenga todas las celdas ejecutadas correctamente.
4. Por favor, nombren el archivo de acuerdo con el siguiente formato `{email}_lab11.ipynb`.
5. Si tienen alguna duda, pueden escribirme a mi correo `j.rayom@uniandes.edu.co` o contactarme directamente por Teams.

---

## Objetivos

1. Aplicar técnicas de procesamiento de lenguaje natural con modelos BERT para resolver una tarea de clasificación de texto multiclase.
2. Comparar el desempeño entre un modelo entrenado desde cero y uno con fine-tuning de un modelo preentrenado.
3. Evaluar modelos con métricas de clasificación.

---

En esta ocasión trabajaremos con el dataset **BBC Full Text Document Classification**, que contiene artículos de noticias de la BBC clasificados en 5 categorías: **business**, **entertainment**, **politics**, **sport** y **tech**.

Dataset: [BBC Full Text Document Classification](https://www.kaggle.com/datasets/alfathterry/bbc-full-text-document-classification)

---

## Instrucciones

### 1. Carga y preparación de datos (10%)

1. Descargue el dataset. La columna `data` contiene los artículos y `labels` es la variable objetivo.
2. Limpie el dataset aplicando preprocesamiento básico de texto (minúsculas, eliminación de caracteres especiales).
3. Separe el dataset en entrenamiento y prueba usando 10% para testing.

---

### 2. Tokenización (10%)

1. Utilice el tokenizador de `bert-base-uncased` para tokenizar los datasets de entrenamiento y prueba.

---

### 3. Definición de arquitectura del modelo (10%)

1. Usando capas densas, agregue una cabeza de clasificación a un modelo BERT.
2. Muestre el resumen del modelo e indique el número total de parámetros.

---

### 4. Entrenamiento desde cero (30%)

1. Parta únicamente de la arquitectura (no use pesos pre-entrenados).
2. Entrene el modelo para predecir las cinco categorías del dataset.
3. Grafique las curvas de pérdida y precisión (entrenamiento vs validación).
4. Valide el modelo usando el dataset de test. Genere un reporte completo de clasificación.

---

### 5. Fine-Tuning con modelo preentrenado (30%)

1. Parta de un modelo BERT pre-entrenado.
2. Entrene el modelo para predecir las cinco categorías del dataset.
3. Grafique las curvas de pérdida y precisión (entrenamiento vs validación).
4. Valide el modelo usando el dataset de test. Genere un reporte completo de clasificación.

---

### 6. Comparación de resultados (10%)

1. Compare el accuracy de ambos modelos en el conjunto de prueba y explique los resultados obtenidos.

---


# **1. Carga y preparación de los datos**

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import re

In [ ]:
bbc_data = pd.read_csv("../datasets/bbc_data.csv")

In [ ]:
bbc_data.head()

In [ ]:
print("ROWS = ", bbc_data.shape[0])
print("COLUMNS = ", bbc_data.shape[1])

In [ ]:
total_count = bbc_data["labels"].value_counts()

print(total_count)
print(f"\nNUMBER OF UNIQUE CLASSES = {len(total_count)}")

In [ ]:
def clean_text(text):

    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
bbc_data["clean_data"] = bbc_data["data"].apply(clean_text)
bbc_data.head()

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    bbc_data["clean_data"], bbc_data["labels"],
    test_size = 0.10,
    random_state = 42,
    stratify = bbc_data["labels"]
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size = (10 / 90),
    random_state = 42,
    stratify = y_temp
)

In [ ]:
print(f"TRAIN = {len(X_train)} samples")
print(f"VALIDATION = {len(X_val)} samples")
print(f"TEST = {len(X_test)} samples")

# **2. Tokenización y preparación de los datos**

In [ ]:
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
print(f"VOCABULARY SIZE = {tokenizer.vocab_size}")

In [ ]:
le = LabelEncoder()
le.fit(bbc_data["labels"])

In [ ]:
y_train_enc = le.transform(y_train)
y_val_enc = le.transform(y_val)
y_test_enc = le.transform(y_test)

In [ ]:
print("CLASSES = ", list(le.classes_))

In [ ]:
MAX_LEN = 512

In [ ]:
def tokenize(texts):
    return tokenizer(
        list(texts),
        padding = True,
        truncation = True,
        max_length = MAX_LEN,
        return_tensors = "pt"
    )

In [ ]:
train_encodings = tokenize(X_train)
val_encodings = tokenize(X_val)
test_encodings = tokenize(X_test)

In [ ]:
print("TRAIN = ", train_encodings["input_ids"].shape)
print("VALIDATION = ", val_encodings["input_ids"].shape)
print("TEST = ", test_encodings["input_ids"].shape)

# **3. Definición de la arquitectura del modelo**

In [ ]:
from transformers import BertModel
import torch.nn as nn
import torch

In [ ]:
class BertClassifier(nn.Module):

    def __init__(self, bert_model, num_classes = 5, dropout = 0.3):
        
        super().__init__()

        self.bert = bert_model
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(self.bert.config.hidden_size, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(input_ids = input_ids, attention_mask = attention_mask)
        pooled = outputs.pooler_output

        x = self.dropout(pooled)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [ ]:
bert = BertModel.from_pretrained("bert-base-uncased")
model = BertClassifier(bert)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)

In [ ]:
print(f"TOTAL PARAMETERS = {total_params:,}")
print(f"TRAINABLE PARAMETERS = {trainable_params:,}")

# **4. Entrenamiento del modelo desde cero**

In [ ]:
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
from transformers import BertConfig
import matplotlib.pyplot as plt
from torch.optim import Adam

In [ ]:
class BBCDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype = torch.long)
        }

In [ ]:
BATCH_SIZE = 24

In [ ]:
train_dataset = BBCDataset(train_encodings, y_train_enc)
val_dataset = BBCDataset(val_encodings, y_val_enc)
test_dataset = BBCDataset(test_encodings, y_test_enc)

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE)

In [ ]:
print(f"TRAIN BATCHES = {len(train_loader)}")
print(f"VALIDATION BATCHES = {len(val_loader)}")
print(f"TEST BATCHES = {len(test_loader)}")

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE = ", DEVICE)

In [ ]:
config = BertConfig()
bert_scratch = BertModel(config)
model_scratch = BertClassifier(bert_scratch).to(DEVICE)

In [ ]:
EPOCHS = 20
LR = 2e-4

In [ ]:
optimizer = Adam(model_scratch.parameters(), lr = LR)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):

    model.train()
    total_loss, correct = 0, 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(dim = 1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

In [ ]:
def eval_epoch(model, loader, criterion, device):

    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            correct += (outputs.argmax(dim = 1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

In [ ]:
history_scratch = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

In [ ]:
for epoch in range(1, EPOCHS + 1):

    train_loss, train_acc = train_epoch(model_scratch, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc = eval_epoch(model_scratch,  val_loader,   criterion, DEVICE)

    history_scratch["train_loss"].append(train_loss)
    history_scratch["val_loss"].append(val_loss)
    history_scratch["train_acc"].append(train_acc)
    history_scratch["val_acc"].append(val_acc)

    print(f"EPOCH {epoch}/{EPOCHS} | TRAIN LOSS = {train_loss:.4f} | TRAIN ACC = {train_acc:.4f} | VAL LOSS = {val_loss:.4f} | VAL ACC = {val_acc:.4f}")

In [ ]:
epochs = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (12, 4))

ax1.plot(epochs, history_scratch["train_loss"], label = "TRAIN")
ax1.plot(epochs, history_scratch["val_loss"],   label = "VALIDATION")
ax1.set_title("LOSS")
ax1.set_xlabel("EPOCH")
ax1.legend()

ax2.plot(epochs, history_scratch["train_acc"], label = "TRAIN")
ax2.plot(epochs, history_scratch["val_acc"],   label = "VALIDATION")
ax2.set_title("ACCURACY")
ax2.set_xlabel("EPOCH")
ax2.legend()

plt.suptitle("TRAINING FROM SCRATCH")
plt.tight_layout()
plt.show()

In [ ]:
model_scratch.eval()
all_preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        outputs = model_scratch(input_ids, attention_mask)
        preds   = outputs.argmax(dim = 1).cpu().numpy()
        all_preds.extend(preds)

In [ ]:
print(classification_report(y_test_enc, all_preds, target_names = le.classes_))

# **5. Fine-tuning con un modelo pre-entrenado**

In [ ]:
bert_pretrained = BertModel.from_pretrained("bert-base-uncased")
model_finetune = BertClassifier(bert_pretrained).to(DEVICE)

print("MODEL LOADED ON DEVICE = ", DEVICE)

In [ ]:
LR_FT = 2e-5

optimizer_ft = Adam(model_finetune.parameters(), lr = LR_FT)

In [ ]:
history_finetune = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

In [ ]:
for epoch in range(1, EPOCHS + 1):

    train_loss, train_acc = train_epoch(model_finetune, train_loader, optimizer_ft, criterion, DEVICE)
    val_loss, val_acc = eval_epoch(model_finetune,  val_loader,   criterion, DEVICE)

    history_finetune["train_loss"].append(train_loss)
    history_finetune["val_loss"].append(val_loss)
    history_finetune["train_acc"].append(train_acc)
    history_finetune["val_acc"].append(val_acc)

    print(f"EPOCH {epoch}/{EPOCHS} | TRAIN LOSS = {train_loss:.4f} | TRAIN ACC = {train_acc:.4f} | VAL LOSS = {val_loss:.4f} | VAL ACC = {val_acc:.4f}")

In [ ]:
epochs = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (12, 4))

ax1.plot(epochs, history_finetune["train_loss"], label = "TRAIN")
ax1.plot(epochs, history_finetune["val_loss"],   label = "VALIDATION")
ax1.set_title("LOSS")
ax1.set_xlabel("EPOCH")
ax1.legend()

ax2.plot(epochs, history_finetune["train_acc"], label = "TRAIN")
ax2.plot(epochs, history_finetune["val_acc"],   label = "VALIDATION")
ax2.set_title("ACCURACY")
ax2.set_xlabel("EPOCH")
ax2.legend()

plt.suptitle("FINE-TUNING WITH PRE-TRAINED MODEL")
plt.tight_layout()
plt.show()

In [ ]:
model_finetune.eval()
all_preds_ft = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        outputs = model_finetune(input_ids, attention_mask)
        preds   = outputs.argmax(dim = 1).cpu().numpy()
        all_preds_ft.extend(preds)

print(classification_report(y_test_enc, all_preds_ft, target_names = le.classes_))